# Výzva: Analýza textu o datové vědě

V tomto příkladu si uděláme jednoduché cvičení, které pokryje všechny kroky tradičního procesu datové vědy. Nemusíte psát žádný kód, stačí kliknout na buňky níže, spustit je a pozorovat výsledek. Jako výzvu máte možnost vyzkoušet tento kód s různými daty.

## Cíl

V této lekci jsme diskutovali různé koncepty související s datovou vědou. Zkusme objevit další související koncepty provedením **textového dolování**. Začneme textem o datové vědě, z něj extrahujeme klíčová slova a pak se pokusíme vizualizovat výsledek.

Jako text použiji stránku o datové vědě z Wikipedie:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Krok 1: Získání dat

Prvním krokem v každém procesu datové vědy je získání dat. Použijeme k tomu knihovnu `requests`:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Krok 2: Transformace dat

Dalším krokem je převést data do podoby vhodné pro zpracování. V našem případě jsme si stáhli zdrojový kód HTML ze stránky a potřebujeme ho převést na prostý text.

Existuje mnoho způsobů, jak toho dosáhnout. Použijeme [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), populární knihovnu v Pythonu pro parsování HTML. BeautifulSoup nám umožňuje cílit na konkrétní HTML prvky, takže se můžeme zaměřit na hlavní obsah článku z Wikipedie a snížit množství navigačních menu, postranních panelů, zápatí a jiného irelevantního obsahu (i když nějaký text šablony může zůstat).


Nejprve je potřeba nainstalovat knihovnu BeautifulSoup pro analýzu HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Krok 3: Získávání poznatků

Nejdůležitějším krokem je převést naše data do podoby, ze které můžeme získat poznatky. V našem případě chceme z textu extrahovat klíčová slova a zjistit, která klíčová slova jsou smysluplnější.

Použijeme Python knihovnu nazvanou [RAKE](https://github.com/aneesha/RAKE) pro extrakci klíčových slov. Nejprve si tuto knihovnu nainstalujeme, pokud ještě není přítomna: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Hlavní funkce je dostupná z objektu `Rake`, který můžeme upravit pomocí některých parametrů. V našem případě nastavíme minimální délku klíčového slova na 5 znaků, minimální frekvenci klíčového slova v dokumentu na 3 a maximální počet slov v klíčovém slově na 2. Klidně si pohrávejte s dalšími hodnotami a sledujte výsledek.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Získali jsme seznam termínů spolu s jejich přiřazeným stupněm důležitosti. Jak vidíte, nejrelevantnější disciplíny, jako je strojové učení a velká data, jsou v seznamu na předních pozicích.

## Krok 4: Vizualizace výsledku

Lidé nejlépe interpretují data ve vizuální podobě. Často tedy dává smysl data vizualizovat, abychom z nich mohli vyvodit nějaké závěry. Můžeme použít knihovnu `matplotlib` v Pythonu pro zobrazení jednoduché distribuce klíčových slov s jejich relevancí:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Existuje však ještě lepší způsob, jak vizualizovat frekvence slov – pomocí **Word Cloud**. Budeme potřebovat nainstalovat další knihovnu pro vykreslení word cloudu z našeho seznamu klíčových slov.


In [ ]:
!{sys.executable} -m pip install wordcloud

Objekt `WordCloud` je zodpovědný za přijetí buď původního textu, nebo předvýpočtového seznamu slov s jejich frekvencemi, a vrací obrázek, který pak může být zobrazen pomocí `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Můžeme také předat originální text do `WordCloud` – uvidíme, jestli dokážeme získat podobný výsledek:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Můžete vidět, že slovní oblak nyní vypadá působivěji, ale také obsahuje spoustu šumu (např. nesouvisející slova jako `Retrieved on`). Také získáváme méně klíčových slov, která se skládají ze dvou slov, jako *data scientist* nebo *computer science*. To je způsobeno tím, že algoritmus RAKE lépe vybírá dobrá klíčová slova z textu. Tento příklad ilustruje důležitost předzpracování a čištění dat, protože jasný obraz na konci nám umožní činit lepší rozhodnutí.

V tomto cvičení jsme prošli jednoduchý proces extrakce významu z textu Wikipedie ve formě klíčových slov a slovního oblaku. Tento příklad je poměrně jednoduchý, ale dobře ukazuje všechny typické kroky, které datový vědec podnikne při práci s daty, počínaje sběrem dat až po vizualizaci.

V našem kurzu budeme všechny tyto kroky podrobně probírat.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
